# Suspected Condition: SNOMED CT in the EPR, HPO on the Order Form

This is the thirteenth notebook in the series. An NHS Trust EPR records a patient's
suspected/diagnosed condition as a SNOMED CT code - that's the national standard for
problem lists and diagnoses. NW-GMSA's rare disease ordering pathway wants something
different: [`Questionnaire-GMSWGSRareDisease.html`](https://nw-gmsa.github.io/en/Questionnaire-GMSWGSRareDisease.html)
requires **HPO** (Human Phenotype Ontology) terms for the patient's phenotype, and a
bespoke NW-GMSA code (`CodeSystem-GenomicClinicalIndication`) for the clinical
indication. Neither is something an EPR produces natively - both need converting from
whatever SNOMED CT code the EPR already holds.

Genomics England publishes a public terminology server for exactly this kind of
conversion:
[re-docs.genomicsengland.co.uk/terminology_server](https://re-docs.genomicsengland.co.uk/terminology_server).
This notebook uses it live against real SNOMED CT codes to find out - honestly, not
just by reading the docs - where that conversion actually works, and where it
doesn't.

## The Questionnaire's HPO requirement

[`Questionnaire-GMSWGSRareDisease.html`](https://nw-gmsa.github.io/en/Questionnaire-GMSWGSRareDisease.html):

- **`NOS/TestDirectoryClinicalIndication-rd`** (choice, `1..1`) - the Genomic Test
  Codes value set, `ServiceRequest.code`. Fills a gap the common core's own Test Code
  item doesn't cover (Whole Genome Sequencing).
- **`NOS/SpecificRareDiseaseSuspected`** (string, `0..1`) - free text,
  `ServiceRequest.reasonCode`. A human-readable complement to the coded items, not a
  replacement.
- **`HPOTerms`** (group, `1..1`, mandatory) - *"at least one HPO term required; WGS
  analysis cannot commence without them"*.
  - **`NOS/HPOTerm`** (open-choice, `1..*`) → `Condition.code`. Bound to "GMS WGS Rare
    Disease Form - Guide HPO Terms" - 38 example terms, explicitly non-exhaustive. The
    IG itself notes: *"No confirmed FHIR CodeSystem binding exists in this IG for the
    Human Phenotype Ontology itself
    (canonically `http://purl.obolibrary.org/obo/hp.owl`) yet"*.
  - **`NOS/HPOTermStatus`** (choice, `1..*`) → `Condition.verificationStatus` - Present
    / Absent / Unknown, LOINC `LA33-6`/`LA32-8`/`LA4489-6`.

`HPOTerm` being `open-choice` matters: it accepts a value from the guide list *or* a
free-entered code, since 38 examples can't cover every rare disease phenotype. Either
way, something has to decide *which* HPO term(s) apply to a given patient - and an
EPR's problem list holds SNOMED CT, not HPO.

In [1]:
import requests

ONTOSERVER = "https://ontoserver.aws.gel.ac/fhir"
SNOMED_SYSTEM = "http://snomed.info/sct"
HPO_SYSTEM = "http://purl.obolibrary.org/obo/hp.owl"


def lookup_snomed(code):
    """Confirm a SNOMED CT code is real, and get its actual display term - $lookup,
    not memory. sct-to-hpo's own $translate gives the same "no mapping" message for a
    made-up code as for a real one with no mapping, so this is the only way to tell
    the two apart."""
    r = requests.get(f"{ONTOSERVER}/CodeSystem/$lookup", params={"system": SNOMED_SYSTEM, "code": code})
    data = r.json()
    if data["resourceType"] == "OperationOutcome":
        return None
    return next(p["valueString"] for p in data["parameter"] if p["name"] == "display")


def translate_sct_to_hpo(code):
    r = requests.get(
        f"{ONTOSERVER}/ConceptMap/sct-to-hpo/$translate",
        params={"conceptMapVersion": "1.0.0", "system": SNOMED_SYSTEM, "code": code, "target": HPO_SYSTEM},
    )
    data = r.json()
    result = next(p["valueBoolean"] for p in data["parameter"] if p["name"] == "result")
    if not result:
        return None
    match = next(p for p in data["parameter"] if p["name"] == "match")
    concept = next(p["valueCoding"] for p in match["part"] if p["name"] == "concept")
    return concept["code"], concept["display"]


print(requests.get(f"{ONTOSERVER}/metadata").status_code, "- no auth needed, unlike this repo's own FHIR_SERVER")

200 - no auth needed, unlike this repo's own FHIR_SERVER


## Where SNOMED already looks like HPO: signs and findings

SNOMED CT codes for individual clinical *signs* - the kind of thing you'd actually
observe or elicit on examination - translate cleanly. Codes below confirmed real via
`$lookup` first, not assumed from memory.

In [2]:
FINDING_CODES = ["20262006", "91175000", "398152000", "110359009"]

for sct_code in FINDING_CODES:
    display = lookup_snomed(sct_code)
    mapping = translate_sct_to_hpo(sct_code)
    print(f"SNOMED {sct_code} {display!r:28} -> {mapping if mapping else 'NO MAPPING'}")

SNOMED 20262006 'Ataxia'                     -> ('HP:0001251', 'Ataxia')
SNOMED 91175000 'Seizure'                    -> ('HP:0001250', 'Seizures')


SNOMED 398152000 'Poor muscle tone'           -> ('HP:0001252', 'Muscular hypotonia')
SNOMED 110359009 'Learning disability'        -> ('HP:0001249', 'Intellectual disability')


## Where it doesn't: named diseases and syndromes

A *diagnosis* is a different kind of thing to a *phenotype*. `Questionnaire-GMSWGSRareDisease.html`'s
`HPOTerm` item wants the latter - HPO is built to describe observed clinical features,
not to name diseases. Three real, `$lookup`-confirmed SNOMED disease/syndrome codes,
translated the same way as above:

In [3]:
DISEASE_CODES = ["19346006", "190905008", "716318002"]  # Marfan's syndrome, Cystic fibrosis, Lynch syndrome

for sct_code in DISEASE_CODES:
    display = lookup_snomed(sct_code)
    mapping = translate_sct_to_hpo(sct_code)
    print(f"SNOMED {sct_code} {display!r:28} -> {mapping if mapping else 'NO MAPPING'}")

SNOMED 19346006 "Marfan's syndrome"          -> NO MAPPING
SNOMED 190905008 'Cystic fibrosis'            -> NO MAPPING


SNOMED 716318002 'Lynch syndrome'             -> NO MAPPING


None of the three map. Not a coverage gap in this particular sample - it's what the
`sct-to-hpo` `ConceptMap` is actually for: a disease name doesn't correspond to a
single phenotypic abnormality, so there's nothing sensible for a `$translate` call to
return. `Marfan's syndrome` alone implies dozens of HPO terms (`HP:0001519` "Disproportionate
tall stature", `HP:0001166` "Arachnodactyly", `HP:0001083` "Ectopia lentis", ...) -
picking the *right* subset for a given patient needs clinical knowledge of what's
actually present, which a `$translate` call over a single diagnosis code can't supply.

Worth being honest about coverage too, not just this disease/finding split: querying a
handful of codes here isn't the same as confirming the map is comprehensive even for
genuine findings - check any code actually needed in practice against the live server
rather than assuming it's covered because it "sounds like" a sign.

### What this means for `NOS/HPOTerm`

If an EPR's problem list already records the patient's *observed findings* as discrete
SNOMED CT codes (poor muscle tone, seizures, ...), each one is a reasonable candidate
for `$translate` to turn into an `NOS/HPOTerm` answer directly. If all the EPR holds is
a *diagnosis* - "suspected Marfan syndrome" - there's no automatic route to the HPO
terms the order actually needs; a clinician has to supply the specific features present
in this patient, which is presumably why the Questionnaire's own guide value set is
described as a non-exhaustive *example* list rather than something meant to be
derived automatically.

## A similar issue: `CodeSystem-GenomicClinicalIndication`

[`CodeSystem-GenomicClinicalIndication.html`](https://nw-gmsa.github.io/en/CodeSystem-GenomicClinicalIndication.html)
is a bespoke NW-GMSA code system - "a fragment" mapped to 1st-level Genomic Test
Directory codes: `R125` "Thoracic aortic aneurysm or dissection", `R361` "Childhood
onset hereditary spastic paraplegia", `R210` "Inherited MMR deficiency (Lynch
syndrome)", and more. [`hl7v2.html`](https://nw-gmsa.github.io/en/hl7v2.html) gives the
worked example for where it's actually used - `OBR-31` (Reason for Study):
`R210^Lynch syndrome^GenomicClinicalIndication`.

Compare that to `DG1-3` (Diagnosis Code) a few fields earlier on the same page - "the
coded diagnosis", mapped to `ServiceRequest.reasonCode`/`Condition`, and (per NHS
practice) SNOMED CT. Same shape as the `HPOTerm` problem: the EPR's own diagnosis code
for Lynch syndrome is SNOMED, but `OBR-31` needs `GenomicClinicalIndication`'s `R210`
instead - a second field on the same order needing the same kind of conversion the HPO
terms did.

In [4]:
lynch_hpo = translate_sct_to_hpo("716318002")
print("SNOMED 716318002 (Lynch syndrome) -> HPO:", lynch_hpo if lynch_hpo else "NO MAPPING")

r = requests.get(f"{ONTOSERVER}/ConceptMap", params={"_summary": "true", "_count": 50})
concept_maps = r.json()
print()
print(f"{concept_maps['total']} ConceptMap(s) published on this terminology server:")
for entry in concept_maps.get("entry", []):
    resource = entry["resource"]
    print(" -", resource["id"], "|", resource.get("name") or resource.get("title"))

SNOMED 716318002 (Lynch syndrome) -> HPO: NO MAPPING

4 ConceptMap(s) published on this terminology server:
 - sct-to-opcs | SNOMED CT UK to OPCS Map
 - sct-to-hpo | SNOMED CT to HPO Map
 - sct-to-icd10 | SNOMED CT UK to ICD-10 Map
 - 806233a2-dd42-4816-9198-4bedb3885541 | SNOMED CT UK to ICD-10 Map


No `sct-to-genomic-clinical-indication` map, or anything close to it - the whole
server only publishes four `ConceptMap`s, none targeting `GenomicClinicalIndication`
or the Genomic Test Directory. Where the HPO problem at least has a real (if narrow)
terminology-server route to try, this one doesn't appear to have an automated route at
all - today, going from an EPR's SNOMED diagnosis to the right `R`-code looks like it
depends on clinical judgement, or NHS England's own published National Genomic Test
Directory eligibility-criteria documentation (a document, not a live API), rather than
anything callable.

## Summary

- Both problems share a shape: the EPR's own coding (SNOMED CT) doesn't match what the
  genomic order needs (`HPOTerm`'s HPO, `OBR-31`'s `GenomicClinicalIndication`), so
  something has to convert between them.
- Genomics England's terminology server
  ([re-docs.genomicsengland.co.uk/terminology_server](https://re-docs.genomicsengland.co.uk/terminology_server),
  `https://ontoserver.aws.gel.ac/fhir`, no auth required) genuinely helps with the HPO
  half, via `ConceptMap/sct-to-hpo/$translate` - but only for SNOMED codes that are
  themselves clinical *signs/findings*. A *diagnosis* code (a named disease or
  syndrome) reliably returns no mapping, confirmed against three real examples
  (Marfan's syndrome, Cystic fibrosis, Lynch syndrome) - not a gap to work around, but
  the actual shape of the problem: a diagnosis doesn't correspond to one phenotype.
- The practical consequence for `NOS/HPOTerm`: automatic translation only helps when
  the EPR already records discrete observed findings, not just a diagnosis label -
  otherwise a clinician has to pick the applicable HPO terms directly, which is likely
  why the Questionnaire's own guide list is deliberately non-exhaustive.
- `GenomicClinicalIndication` has the same underlying problem (`OBR-31`'s own worked
  example, `R210^Lynch syndrome`, uses a SNOMED-diagnosable condition) but no
  equivalent terminology-server `ConceptMap` exists to help with it at all - confirmed
  by listing every `ConceptMap` the server publishes, not assumed from its absence in
  the docs.
- Every call in this notebook hit the real, public Genomics England terminology
  server live - no vendored/cached responses, and no NW-GMSA `V2_TOOLS`/`V2_SERVER`
  connection needed either.